[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C54_DETR_Set_Prediction_Course/02_set_loss/02_set_prediction_loss.ipynb)

# 02 · 集合预测损失（IoU 家族 / 解析梯度 / Hungarian loss / no-object 权重）

目标：把 DETR 的集合损失**从零手写一遍**，并用数值梯度证明你推的公式是对的。

**本 notebook 你会亲手实现：**
1. `IoU / GIoU / DIoU / CIoU` —— 四个函数，从坐标算起，含全部退化情形保护
2. **它们的解析梯度**，并用中心差分数值梯度逐项校验（白板高频题）
3. **IoU 梯度死区实验**：把预测框推开，看 IoU 梯度精确变成 0 而 GIoU 不会
4. **L1 的尺度敏感性**：同样 2 px 偏移，8×8 框与 256×256 框的 L1 完全相同、1−IoU 差 20 倍
5. **匈牙利匹配 + 完整 Hungarian loss**（分类 CE + L1 + GIoU + no-object）
6. **no-object 权重的消融**：在合成数据上看前景召回随 `eos_coef` 变化
7. 四道练习：向量化 IoU 矩阵 / DETR 代价矩阵 / 辅助损失与层间翻转率 / cxcywh 参数化的梯度

> 心智模型：**L1 把框拽到大致位置（快但对尺度不公平），
> GIoU 把框调到形状贴合（尺度公平但远处梯度弱）。缺一个都不行。**

## 1 · IoU 从零实现：三个必查点

① `U = Aa + Ab - I`（不是 `Aa + Ab`）
② 交集宽高必须 clamp 到 ≥0（**不 clamp 时两个负数相乘会得到正的假交集**）
③ 除零保护

In [ ]:
import numpy as np, math, itertools, json
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(0)

EPS = 1e-12

def box_wh(b):
    '''b = (x1, y1, x2, y2) -> (w, h)'''
    return b[2] - b[0], b[3] - b[1]

def inter_union(a, b):
    '''返回 (I, U, Aa, Ab, iw, ih)。iw/ih 已 clamp。'''
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)   # ← ② clamp，缺了就有「假交集」
    I = iw * ih
    Aa = (a[2] - a[0]) * (a[3] - a[1])
    Ab = (b[2] - b[0]) * (b[3] - b[1])
    U = Aa + Ab - I                                      # ← ① 减掉交集
    return I, U, Aa, Ab, iw, ih

def iou(a, b):
    I, U, *_ = inter_union(a, b)
    return I / (U + EPS)                                 # ← ③ 除零保护

# --- 自校验 ---
g = (0.0, 0.0, 8.0, 8.0)
assert abs(iou(g, g) - 1.0) < 1e-9, '同一个框 IoU 必须是 1'
assert iou(g, (100., 100., 108., 108.)) == 0.0, '完全不相交必须是 0'

# 规范里的关键数字：8x8 的框对角偏移 2 px
shift = (2.0, 2.0, 10.0, 10.0)
I, U, *_ = inter_union(g, shift)
print('8x8 框对角偏移 2px:  I=%.0f  U=%.0f  IoU=%.4f' % (I, U, I / U))
assert abs(iou(g, shift) - 36 / 92) < 1e-9, 'I=6*6=36, U=64+64-36=92'

# 对照：64x64 的框同样偏移 2px
g64 = (0., 0., 64., 64.); s64 = (2., 2., 66., 66.)
print('64x64 框对角偏移 2px: IoU=%.4f' % iou(g64, s64))
assert abs(iou(g64, s64) - 3844 / 4348) < 1e-9

# ② 的反面教材：不 clamp 会怎样
def iou_buggy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    I = (ix2 - ix1) * (iy2 - iy1)          # ← 没有 clamp！
    Aa = (a[2]-a[0])*(a[3]-a[1]); Ab = (b[2]-b[0])*(b[3]-b[1])
    return I / (Aa + Ab - I)

far = (14., 14., 22., 22.)          # 与 g 完全不相交（间隔 6 px）
print('\n完全不相交的两个框:  正确 IoU=%.4f   有 bug 的 IoU=%.4f  ← **假交集**'
      % (iou(g, far), iou_buggy(g, far)))
assert iou_buggy(g, far) > 0.3, '两个负数相乘造出了一个虚假的高 IoU'
print('    有 bug 的实现给出 0.391 —— 和「重叠良好的 8x8 框偏移 2px」**一模一样**。')
print('⚠️  这个 bug 只在**完全不相交**时触发 —— 如果单元测试只测相交情形，永远查不出来。')
print('✅ 而在 DETR 训练初期，随机初始化的框和 GT **绝大多数是不相交的**。')

## 2 · GIoU / DIoU / CIoU：每一个都补上一个具体的洞

In [ ]:
def enclosing(a, b):
    '''最小外接框 C 及其宽高、面积、对角线平方。'''
    cx1, cy1 = min(a[0], b[0]), min(a[1], b[1])
    cx2, cy2 = max(a[2], b[2]), max(a[3], b[3])
    cw, ch = cx2 - cx1, cy2 - cy1
    return cx1, cy1, cx2, cy2, cw, ch, cw * ch, cw * cw + ch * ch

def giou(a, b):
    I, U, *_ = inter_union(a, b)
    *_, Ac, _ = enclosing(a, b)
    return I / (U + EPS) - (Ac - U) / (Ac + EPS)

def center(b):
    return (b[0] + b[2]) / 2.0, (b[1] + b[3]) / 2.0

def diou(a, b):
    I, U, *_ = inter_union(a, b)
    *_, _, dc2 = enclosing(a, b)
    (cax, cay), (cbx, cby) = center(a), center(b)
    rho2 = (cax - cbx) ** 2 + (cay - cby) ** 2
    return I / (U + EPS) - rho2 / (dc2 + EPS)

def aspect_v(a, b):
    wa, ha = box_wh(a); wb, hb = box_wh(b)
    d = math.atan2(wb, hb) - math.atan2(wa, ha)
    return (4.0 / math.pi ** 2) * d * d

def ciou(a, b):
    v = aspect_v(a, b)
    i = iou(a, b)
    alpha = v / (1.0 - i + v + EPS)          # 实现里 alpha 通常被 detach
    return diou(a, b) - alpha * v

# --- 性质自校验 ---
for _ in range(200):
    a = np.sort(rng.uniform(-3, 3, 2)); c = np.sort(rng.uniform(-3, 3, 2))
    b = np.sort(rng.uniform(-3, 3, 2)); d = np.sort(rng.uniform(-3, 3, 2))
    A = (a[0], b[0], a[1] + 0.1, b[1] + 0.1)
    B = (c[0], d[0], c[1] + 0.1, d[1] + 0.1)
    i_, g_, di_, ci_ = iou(A, B), giou(A, B), diou(A, B), ciou(A, B)
    assert -1.0 - 1e-9 <= g_ <= i_ + 1e-9, 'GIoU <= IoU 且 >= -1'
    assert di_ <= i_ + 1e-9, 'DIoU <= IoU（多减了一个非负的中心距离项）'
    assert ci_ <= di_ + 1e-9, 'CIoU <= DIoU（多减了一个非负的宽高比项）'

print('%-34s %8s %8s %8s %8s' % ('配置', 'IoU', 'GIoU', 'DIoU', 'CIoU'))
CASES = [
    ('完全重合',              (0,0,4,4),  (0,0,4,4)),
    ('相邻但不相交',          (0,0,4,4),  (5,0,9,4)),
    ('**很远的不相交**',      (0,0,4,4),  (40,0,44,4)),
    ('**包含：小框在中心**',  (0,0,10,10),(4,4,6,6)),
    ('**包含：小框贴左上**',  (0,0,10,10),(0,0,2,2)),
    ('中心重合、宽高比不同',  (0,0,10,10),(2.5,0,7.5,10)),
]
for name, A, B in CASES:
    A = tuple(map(float, A)); B = tuple(map(float, B))
    print('%-30s %8.4f %8.4f %8.4f %8.4f' % (name, iou(A,B), giou(A,B), diou(A,B), ciou(A,B)))

# 关键性质 1：不相交越远，GIoU 越接近 -1（IoU 恒为 0，毫无区分度）
assert iou((0.,0.,4.,4.), (5.,0.,9.,4.)) == iou((0.,0.,4.,4.), (40.,0.,44.,4.)) == 0.0
assert giou((0.,0.,4.,4.), (40.,0.,44.,4.)) < giou((0.,0.,4.,4.), (5.,0.,9.,4.))
# 关键性质 2：包含关系下 GIoU 退化成 IoU（外接框 C == 大框，惩罚项归零）
A, B1, B2 = (0.,0.,10.,10.), (4.,4.,6.,6.), (0.,0.,2.,2.)
assert abs(giou(A,B1) - iou(A,B1)) < 1e-9 and abs(giou(A,B2) - iou(A,B2)) < 1e-9
assert abs(giou(A,B1) - giou(A,B2)) < 1e-9, 'GIoU 对这两种包含关系**完全无区分度**'
print('\n⚠️  「小框在中心」与「小框贴左上」的 IoU/GIoU **完全相同** —— GIoU 分不出来。')
print('    IoU: %.4f vs %.4f   GIoU: %.4f vs %.4f   **DIoU: %.4f vs %.4f  ← 分开了**'
      % (iou(A,B1), iou(A,B2), giou(A,B1), giou(A,B2), diou(A,B1), diou(A,B2)))
assert diou(A,B1) > diou(A,B2) + 1e-3, 'DIoU 的中心距离项恰好补上这个洞'

## 3 · 解析梯度：手推 + 数值校验（白板高频题）

`max/min` 会产生**指示函数**；`clamp(·, 0)` 会产生**截断乘子** `1[iw>0]1[ih>0]` ——
这个乘子就是「不相交时梯度恒为 0」在代码里的样子，也是白板题最容易漏的一项。

In [ ]:
def _grad_I_U(a, b):
    '''返回 (I, U, dI/da, dU/da)，a=(x1,y1,x2,y2) 为变量，b 为常量。'''
    I, U, Aa, Ab, iw, ih = inter_union(a, b)
    if iw > 0 and ih > 0:
        dI = np.array([
            -ih * (a[0] > b[0]),      # d I / d x1   （ix1 = max -> 指示 a.x1 更大）
            -iw * (a[1] > b[1]),      # d I / d y1
            +ih * (a[2] < b[2]),      # d I / d x2   （ix2 = min -> 指示 a.x2 更小）
            +iw * (a[3] < b[3]),      # d I / d y2
        ], dtype=float)
    else:
        dI = np.zeros(4)              # ← **截断：不相交则梯度精确为 0**
    wa, ha = box_wh(a)
    dAa = np.array([-ha, -wa, ha, wa], dtype=float)
    dU = dAa - dI
    return I, U, dI, dU

def grad_iou(a, b):
    I, U, dI, dU = _grad_I_U(a, b)
    return (dI * U - I * dU) / (U * U + EPS)

def grad_giou(a, b):
    I, U, dI, dU = _grad_I_U(a, b)
    g_iou = (dI * U - I * dU) / (U * U + EPS)
    _, _, _, _, cw, ch, Ac, _ = enclosing(a, b)
    dAc = np.array([                  # ← 外接框取 min/max，**指示函数方向与交集相反**
        -ch * (a[0] < b[0]),
        -cw * (a[1] < b[1]),
        +ch * (a[2] > b[2]),
        +cw * (a[3] > b[3]),
    ], dtype=float)
    # GIoU = IoU - 1 + U/Ac
    return g_iou + (dU * Ac - U * dAc) / (Ac * Ac + EPS)

def num_grad(f, a, b, eps=1e-6):
    '''中心差分数值梯度。'''
    a = np.asarray(a, float); out = np.zeros(4)
    for k in range(4):
        ap = a.copy(); ap[k] += eps
        am = a.copy(); am[k] -= eps
        out[k] = (f(tuple(ap), b) - f(tuple(am), b)) / (2 * eps)
    return out

# --- 逐项校验：随机框（tie 是零测集，不会踩到 max/min 的不可导点）---
bad = 0
for _ in range(300):
    a = (float(rng.uniform(-2, 2)), float(rng.uniform(-2, 2)), 0., 0.)
    a = (a[0], a[1], a[0] + float(rng.uniform(.3, 3)), a[1] + float(rng.uniform(.3, 3)))
    b = (float(rng.uniform(-2, 2)), float(rng.uniform(-2, 2)), 0., 0.)
    b = (b[0], b[1], b[0] + float(rng.uniform(.3, 3)), b[1] + float(rng.uniform(.3, 3)))
    for f, gf in [(iou, grad_iou), (giou, grad_giou)]:
        if not np.allclose(gf(a, b), num_grad(f, a, b), atol=2e-6):
            bad += 1
assert bad == 0, '解析梯度与数值梯度不一致，共 %d 例' % bad
print('✅ 300 组随机框 x 2 个函数：解析梯度与中心差分数值梯度全部吻合（atol=2e-6）')

a0 = (0.0, 0.0, 4.0, 4.0); b0 = (1.0, 1.5, 6.0, 5.0)
print('\n示例 a=%s  b=%s' % (a0, b0))
print('  d IoU /d(x1,y1,x2,y2) 解析 =', grad_iou(a0, b0))
print('  d IoU /d(x1,y1,x2,y2) 数值 =', num_grad(iou, a0, b0))
print('  d GIoU/d(x1,y1,x2,y2) 解析 =', grad_giou(a0, b0))
print('  d GIoU/d(x1,y1,x2,y2) 数值 =', num_grad(giou, a0, b0))

In [ ]:
# DIoU / CIoU 的解析梯度（CIoU 按标准实现把 alpha 视作常数 detach）
def grad_diou(a, b):
    I, U, dI, dU = _grad_I_U(a, b)
    g_iou = (dI * U - I * dU) / (U * U + EPS)
    _, _, _, _, cw, ch, _, dc2 = enclosing(a, b)
    (cax, cay), (cbx, cby) = center(a), center(b)
    rho2 = (cax - cbx) ** 2 + (cay - cby) ** 2
    # d rho2 / d x1 = 2(cax-cbx) * d cax/d x1 = 2(cax-cbx) * 0.5
    drho2 = np.array([(cax - cbx), (cay - cby), (cax - cbx), (cay - cby)], float)
    ddc2 = np.array([                          # dc2 = cw^2 + ch^2
        -2 * cw * (a[0] < b[0]),
        -2 * ch * (a[1] < b[1]),
        +2 * cw * (a[2] > b[2]),
        +2 * ch * (a[3] > b[3]),
    ], float)
    return g_iou - (drho2 * dc2 - rho2 * ddc2) / (dc2 * dc2 + EPS)

def grad_ciou(a, b, detach_alpha=True):
    wa, ha = box_wh(a); wb, hb = box_wh(b)
    v = aspect_v(a, b)
    i = iou(a, b)
    alpha = v / (1.0 - i + v + EPS)
    d = math.atan2(wb, hb) - math.atan2(wa, ha)
    # dv/dwa = -(8/pi^2) d * ha/(wa^2+ha^2) ;  dv/dha = +(8/pi^2) d * wa/(wa^2+ha^2)
    k = 8.0 / math.pi ** 2 * d / (wa * wa + ha * ha + EPS)
    dv_dw, dv_dh = -k * ha, k * wa
    dv = np.array([-dv_dw, -dv_dh, dv_dw, dv_dh], float)   # w = x2-x1, h = y2-y1
    g = grad_diou(a, b) - alpha * dv
    if not detach_alpha:                                   # 完整梯度还要过 alpha 里的 IoU
        dalpha_di = v / (1.0 - i + v + EPS) ** 2
        g = g - v * dalpha_di * grad_iou(a, b)
        dalpha_dv = (1.0 - i) / (1.0 - i + v + EPS) ** 2
        g = g - v * dalpha_dv * dv
    return g

bad = 0
for _ in range(300):
    a = (float(rng.uniform(-2, 2)), float(rng.uniform(-2, 2)), 0., 0.)
    a = (a[0], a[1], a[0] + float(rng.uniform(.3, 3)), a[1] + float(rng.uniform(.3, 3)))
    b = (float(rng.uniform(-2, 2)), float(rng.uniform(-2, 2)), 0., 0.)
    b = (b[0], b[1], b[0] + float(rng.uniform(.3, 3)), b[1] + float(rng.uniform(.3, 3)))
    if not np.allclose(grad_diou(a, b), num_grad(diou, a, b), atol=2e-6):
        bad += 1
    if not np.allclose(grad_ciou(a, b, detach_alpha=False), num_grad(ciou, a, b), atol=2e-5):
        bad += 1
assert bad == 0, 'DIoU/CIoU 梯度校验失败 %d 例' % bad
print('✅ DIoU 与 CIoU（完整梯度版）同样通过数值校验')

a0 = (0.0, 0.0, 4.0, 2.0); b0 = (0.5, 0.2, 3.0, 3.5)
print('\nCIoU 在 a=%s b=%s 处：' % (a0, b0))
print('  detach alpha（**标准实现**）:', grad_ciou(a0, b0, True))
print('  完整梯度（数值可校验）      :', grad_ciou(a0, b0, False))
print('  中心差分数值梯度            :', num_grad(ciou, a0, b0))
print('\n⚠️  标准 CIoU 实现里 alpha 被 detach，所以它反传的**不是 CIoU 的真实梯度**。')
print('    这在工程上没问题（alpha 只是个自适应权重），但做数值验证时必须知道，')
print('    否则你会以为自己写错了。')

## 4 · IoU 的梯度死区：把框推开，看梯度精确变成 0

**这是「为什么必须有 GIoU」的直接证据。**

In [ ]:
gt = (0.0, 0.0, 8.0, 8.0)          # 一块 8x8 像素的远处限速牌
print('%8s %8s %10s %14s %14s' % ('平移 dx', 'IoU', 'GIoU', '|grad IoU|', '|grad GIoU|'))
rows = []
for dx in [1.0, 2.0, 4.0, 6.0, 7.9, 8.0, 10.0, 20.0, 50.0]:
    pred = (dx, 0.0, dx + 8.0, 8.0)
    gi = np.abs(grad_iou(pred, gt)).sum()
    gg = np.abs(grad_giou(pred, gt)).sum()
    rows.append((dx, iou(pred, gt), giou(pred, gt), gi, gg))
    print('%8.1f %8.4f %10.4f %14.6f %14.6f'
          % (dx, iou(pred, gt), giou(pred, gt), gi, gg))

# 断言：一旦不相交（dx >= 8），IoU 梯度**精确为 0**，而 GIoU 仍有梯度
for dx, i_, g_, gi, gg in rows:
    if dx >= 8.0:
        assert gi == 0.0, 'dx=%.1f 时 IoU 梯度必须精确为 0' % dx
        assert gg > 1e-4, 'dx=%.1f 时 GIoU 必须仍有梯度' % dx
print('\n⚠️  dx>=8 之后 IoU 梯度**精确等于 0**（不是「很小」，是数学上的零）——')
print('    优化器在这片 plateau 上完全收不到方向信号。')
print('✅ GIoU 的梯度随距离衰减但**永不为零**：|grad| 从 %.4f 衰减到 %.6f'
      % (rows[5][4], rows[-1][4]))
print('   衰减是因为外接框 Ac 随距离平方增长 —— **所以 L1 也不能去掉**，')
print('   L1 的梯度不随距离衰减，负责「快速把框拽回来」。')

# L1 对照：梯度大小与距离无关
def l1_grad_x(dx):
    return 1.0 if dx > 0 else -1.0
print('\n对照 L1: dx=8 时 |grad|=%.1f, dx=50 时 |grad|=%.1f  ← **完全不衰减**'
      % (abs(l1_grad_x(8.)), abs(l1_grad_x(50.))))

## 5 · L1 的尺度敏感性：为什么「只用 L1」对 TSR 是灾难

DETR 预测归一化坐标 `(cx,cy,w,h) ∈ [0,1]^4`。
**同样 2 像素的对角偏移，L1 完全相同，而实际定位质量（1−IoU）差 20 倍。**

In [ ]:
IMG = 640.0        # 输入分辨率

def px_to_cxcywh(x1, y1, x2, y2, img=IMG):
    '''DETR 预测的就是这个：归一化的 (cx, cy, w, h)。'''
    return np.array([(x1 + x2) / 2 / img, (y1 + y2) / 2 / img,
                     (x2 - x1) / img, (y2 - y1) / img])

print('%-16s %9s %14s %10s %10s %10s'
      % ('目标', '像素尺寸', 'L1(cxcywh)', 'IoU', '1-IoU', '1-GIoU'))
recs = []
for name, s in [('远处限速牌', 8), ('中距离标志', 16), ('近处车辆', 64), ('大型广告牌', 256)]:
    gt_px   = (0.0, 0.0, float(s), float(s))
    pred_px = (2.0, 2.0, float(s) + 2, float(s) + 2)      # 对角偏移 2 px
    l1 = float(np.abs(px_to_cxcywh(*pred_px) - px_to_cxcywh(*gt_px)).sum())
    recs.append((name, s, l1, iou(gt_px, pred_px), 1 - giou(gt_px, pred_px)))
    print('%-12s %9s %14.6f %10.4f %10.4f %10.4f'
          % (name, '%dx%d' % (s, s), l1, iou(gt_px, pred_px),
             1 - iou(gt_px, pred_px), 1 - giou(gt_px, pred_px)))

l1s = [r[2] for r in recs]
assert max(l1s) - min(l1s) < 1e-12, '四个目标的 L1 损失**完全相同**'
assert abs(l1s[0] - 2 * 2 / 640) < 1e-12, 'cx 与 cy 各偏 2/640'
ious = [r[3] for r in recs]
print('\n⚠️  四行的 L1 损失完全相同 (%.6f)，但 1-IoU 从 %.4f 到 %.4f，**差 %.0f 倍**。'
      % (l1s[0], 1 - ious[-1], 1 - ious[0], (1 - ious[0]) / (1 - ious[-1])))
assert (1 - ious[0]) / (1 - ious[-1]) > 15

# 反过来：大目标的绝对误差天然更大 -> 只用 L1 时大目标主导梯度
print('\n反方向的问题：按「相对误差 10%」缩放时，L1 谁大？')
print('%-12s %9s %16s' % ('目标', '像素尺寸', 'L1(相对偏移10%)'))
big_l1 = []
for name, s in [('远处限速牌', 8), ('近处车辆', 64), ('大型广告牌', 256)]:
    d = 0.1 * s
    v = float(np.abs(px_to_cxcywh(d, d, s + d, s + d) - px_to_cxcywh(0, 0, s, s)).sum())
    big_l1.append(v)
    print('%-12s %9s %16.6f' % (name, '%dx%d' % (s, s), v))
assert big_l1[-1] > 30 * big_l1[0], '同样的相对误差，大目标的 L1 大 32 倍 -> 主导梯度'
print('\n✅ 结论（**TSR 场景的核心论证**）：')
print('   · 只用 L1 -> 小目标的定位错误被严重低估，大目标主导梯度；')
print('   · 只用 IoU 系 -> 训练初期框不相交，梯度为 0，根本启动不了；')
print('   · **L1 + GIoU 恰好互补**：L1 提供处处存在的方向，GIoU 提供尺度公平的形状约束。')
print('   · 交通标志绝大多数落在 8-32 px 区间 —— 这正是 L1 最不公平、GIoU 最关键的区间。')

## 6 · 匈牙利匹配 + 完整 Hungarian loss

匹配算法在模块 01 讲过，这里给一个紧凑的 O(n³) 实现（含暴力对拍），
重点是**代价矩阵与损失是两个不同的函数**。

In [ ]:
def hungarian(cost):
    '''O(n^3) 匈牙利算法（JV 势函数版），支持 n <= m 的矩形代价矩阵。
       返回 (row_ind, col_ind)，使总代价最小。'''
    C = np.asarray(cost, dtype=float)
    n, m = C.shape
    assert n <= m, '行数必须 <= 列数'
    INF = float('inf')
    u = np.zeros(n + 1); v = np.zeros(m + 1)
    p = np.zeros(m + 1, dtype=int); way = np.zeros(m + 1, dtype=int)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = np.full(m + 1, INF); used = np.zeros(m + 1, dtype=bool)
        while True:
            used[j0] = True
            i0 = p[j0]; delta = INF; j1 = -1
            for j in range(1, m + 1):
                if not used[j]:
                    cur = C[i0 - 1, j - 1] - u[i0] - v[j]
                    if cur < minv[j]:
                        minv[j] = cur; way[j] = j0
                    if minv[j] < delta:
                        delta = minv[j]; j1 = j
            for j in range(m + 1):
                if used[j]:
                    u[p[j]] += delta; v[j] -= delta
                else:
                    minv[j] -= delta
            j0 = j1
            if p[j0] == 0:
                break
        while j0:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
    col = np.zeros(n, dtype=int)
    for j in range(1, m + 1):
        if p[j]:
            col[p[j] - 1] = j - 1
    return np.arange(n), col

# 与暴力枚举对拍
for trial in range(60):
    n, m = int(rng.integers(1, 5)), int(rng.integers(1, 6))
    n = min(n, m)
    C = rng.uniform(-2, 4, size=(n, m))
    r, c = hungarian(C)
    got = C[r, c].sum()
    best = min(sum(C[i, perm[i]] for i in range(n))
               for perm in itertools.permutations(range(m), n))
    assert abs(got - best) < 1e-9, (C, got, best)
    assert len(set(c.tolist())) == n, '必须是一一对应'
print('✅ 匈牙利算法与暴力枚举对拍通过（60 组随机矩形代价矩阵）')

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def cxcywh_to_xyxy(b):
    cx, cy, w, h = b
    return (cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)

W_CLS, W_L1, W_GIOU = 1.0, 5.0, 2.0        # DETR 默认
EOS_COEF = 0.1

def detr_cost_matrix(logits, boxes, tgt_labels, tgt_boxes):
    '''**匹配代价**：分类项用 -p（概率，有界），不用 -log p。M x N。'''
    prob = softmax(logits, -1)             # (N, K+1)，最后一列是 no-object
    M, N = len(tgt_labels), len(boxes)
    C = np.zeros((M, N))
    for i in range(M):
        for j in range(N):
            c_cls = -prob[j, tgt_labels[i]]
            c_l1 = np.abs(np.asarray(boxes[j]) - np.asarray(tgt_boxes[i])).sum()
            c_gi = 1.0 - giou(cxcywh_to_xyxy(boxes[j]), cxcywh_to_xyxy(tgt_boxes[i]))
            C[i, j] = W_CLS * c_cls + W_L1 * c_l1 + W_GIOU * c_gi
    return C

def hungarian_loss(logits, boxes, tgt_labels, tgt_boxes, eos_coef=EOS_COEF):
    '''**训练损失**：分类项用 -log p，且背景 query 全部参与。'''
    N, Kp1 = logits.shape
    K = Kp1 - 1                            # 最后一类是 no-object
    C = detr_cost_matrix(logits, boxes, tgt_labels, tgt_boxes)
    rows, cols = hungarian(C)              # rows 索引 GT，cols 索引 query
    logp = np.log(softmax(logits, -1) + 1e-12)

    target = np.full(N, K, dtype=int)      # 默认全是 no-object
    weight = np.full(N, eos_coef)
    for gi, qi in zip(rows, cols):
        target[qi] = tgt_labels[gi]; weight[qi] = 1.0

    # 与 PyTorch 的 weighted cross_entropy 一致：sum(w*l) / sum(w)
    l_cls = -(weight * logp[np.arange(N), target]).sum() / weight.sum()
    l_l1 = l_gi = 0.0
    for gi, qi in zip(rows, cols):         # **只对匹配上的 query 算框损失**
        l_l1 += np.abs(np.asarray(boxes[qi]) - np.asarray(tgt_boxes[gi])).sum()
        l_gi += 1.0 - giou(cxcywh_to_xyxy(boxes[qi]), cxcywh_to_xyxy(tgt_boxes[gi]))
    nb = max(len(tgt_labels), 1)
    total = W_CLS * l_cls + W_L1 * l_l1 / nb + W_GIOU * l_gi / nb
    return dict(total=total, cls=l_cls, l1=l_l1 / nb, giou=l_gi / nb,
                match=dict(zip(rows.tolist(), cols.tolist())))

# --- 合成一张 TSR 风格的场景：3 块标志，全部小而偏上 ---
K, N = 5, 20
tgt_boxes = [(0.22, 0.30, 0.030, 0.030),   # 远处限速牌 ~ 19x19 px @640
             (0.55, 0.26, 0.018, 0.018),   # 更远的警告牌 ~ 12x12 px
             (0.80, 0.40, 0.055, 0.055)]   # 较近的指示牌 ~ 35x35 px
tgt_labels = [0, 2, 1]

def jitter(b, s_pos, s_wh, rg=None):
    '''给框加噪声，并保证 w, h 恒为正（否则 GIoU 无定义）。'''
    rg = rg or rng
    out = np.asarray(b, float) + np.concatenate([rg.normal(0, s_pos, 2),
                                                 rg.normal(0, s_wh, 2)])
    out[2:] = np.clip(out[2:], 0.005, None)
    return tuple(out)

pred_boxes = [tuple(rng.uniform([0.05, 0.05, 0.01, 0.01], [0.95, 0.7, 0.12, 0.12]))
              for _ in range(N)]
logits = rng.normal(0, 1.0, size=(N, K + 1))
# 让 3 个 query 明显更靠近对应 GT（模拟训练到一半的状态）
for k, (gi, qi) in enumerate([(0, 3), (1, 11), (2, 17)]):
    pred_boxes[qi] = jitter(tgt_boxes[gi], 0.008, 0.002)
    logits[qi, tgt_labels[gi]] += 3.0

out = hungarian_loss(logits, pred_boxes, tgt_labels, tgt_boxes)
print('匹配结果 (GT -> query):', out['match'])
print('损失分解: cls=%.4f  L1=%.4f  GIoU=%.4f  ->  total=%.4f'
      % (out['cls'], out['l1'], out['giou'], out['total']))
print('加权后各项贡献: cls=%.4f  L1=%.4f  GIoU=%.4f'
      % (W_CLS * out['cls'], W_L1 * out['l1'], W_GIOU * out['giou']))
assert out['match'] == {0: 3, 1: 11, 2: 17}, '刻意放近的 3 个 query 应该被匹配上'
assert out['total'] > 0
print('\n✅ 注意加权后三项在同一个量级 —— 这就是 lambda=(1,5,2) 的**量纲配平**作用。')
print('   若把 lambda_L1 设成 1，L1 项贡献只有 %.4f，在分类项面前几乎没有声音。'
      % (1.0 * out['l1']))

## 7 · no-object 权重消融：`eos_coef` 到底在调什么

合成一个「93% 背景 / 7% 前景」的二分类问题（对应 N=100、M=7 的典型场景），
用加权交叉熵训练一个小分类器，看**前景召回**与**误检**如何随 `eos_coef` 变化。

In [ ]:
def make_query_features(n_fg=70, n_bg=930, seed=1):
    '''合成 query 特征：前景与背景**有重叠**（真实情况就是这样）。'''
    r = np.random.default_rng(seed)
    Xf = r.normal([1.2, 0.0], 1.0, size=(n_fg, 2))
    Xb = r.normal([-0.3, 0.0], 1.0, size=(n_bg, 2))
    X = np.vstack([Xf, Xb])
    y = np.concatenate([np.zeros(n_fg, int), np.ones(n_bg, int)])   # 1 = no-object
    X = np.hstack([X, np.ones((len(X), 1))])                        # bias
    return X, y

def train_weighted_ce(X, y, eos_coef, iters=800, lr=0.5):
    '''2 类 softmax + 对 no-object 类降权的交叉熵，全批量梯度下降。'''
    W = np.zeros((X.shape[1], 2))
    w_sample = np.where(y == 1, eos_coef, 1.0)
    for _ in range(iters):
        P_ = softmax(X @ W, -1)
        onehot = np.zeros_like(P_); onehot[np.arange(len(y)), y] = 1.0
        grad = X.T @ ((P_ - onehot) * w_sample[:, None]) / w_sample.sum()
        W -= lr * grad
    return W

X, y = make_query_features()
print('数据: 前景 %d / 背景 %d  ->  背景占比 %.1f%%'
      % ((y == 0).sum(), (y == 1).sum(), 100 * (y == 1).mean()))
print('\n%-12s %14s %14s %14s' % ('eos_coef', '前景召回', '前景精确率', '误检数(FP)'))
res = {}
for eos in [1.0, 0.5, 0.2, 0.1, 0.05, 0.02]:
    W = train_weighted_ce(X, y, eos)
    pred_fg = (X @ W).argmax(1) == 0
    tp = int((pred_fg & (y == 0)).sum()); fp = int((pred_fg & (y == 1)).sum())
    rec = tp / (y == 0).sum()
    prec = tp / max(tp + fp, 1)
    res[eos] = (rec, prec, fp)
    tag = '   <- DETR 默认' if eos == 0.1 else ''
    print('%-12.2f %13.3f %14.3f %14d%s' % (eos, rec, prec, fp, tag))

assert res[1.0][0] < res[0.1][0], 'eos_coef=1.0 时前景召回必须显著低于 0.1'
assert res[0.02][2] > res[0.1][2], 'eos_coef 越小误检越多'
print('\n⚠️  eos_coef=1.0（不降权）时前景召回只有 %.3f —— 模型学到的最优解是「几乎全说背景」，'
      % res[1.0][0])
print('    因为那样能立刻消掉 93% 的损失。这就是 DETR 不降权时 mAP≈0 的机理。')
print('✅ eos_coef=0.1 把有效背景权重降到 %.1f（等效样本数 %d），召回回到 %.3f。'
      % (0.1, int(0.1 * 930), res[0.1][0]))
print('⚠️  但继续调小到 0.02，误检从 %d 涨到 %d —— **这不是「更好」，只是换了工作点**。'
      % (res[0.1][2], res[0.02][2]))
print('   对 TSR 这类误检代价高度不对称的场景，工作点应该由**标定后的阈值**决定，')
print('   而不是由损失权重偷换。')

## ✏️ 练习 1：向量化的 IoU / GIoU 矩阵

实现 `iou_matrix(A, B)` 与 `giou_matrix(A, B)`：
`A` 是 `(N,4)`、`B` 是 `(M,4)` 的 xyxy 数组，返回 `(N,M)` 矩阵。
**不许用 Python 循环**（用 numpy 广播）。这是代价矩阵的性能瓶颈，实际项目里必须向量化。

In [ ]:
def iou_matrix(A, B):
    # TODO: 用广播算 (N,M) 的 IoU 矩阵
    #  提示: lt = np.maximum(A[:, None, :2], B[None, :, :2])
    #        rb = np.minimum(A[:, None, 2:], B[None, :, 2:])
    raise NotImplementedError

def giou_matrix(A, B):
    # TODO: 在 iou_matrix 基础上加最小外接框的惩罚项
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
A = np.array([[0., 0., 4., 4.], [5., 5., 9., 9.], [0., 0., 10., 10.]])
B = np.array([[0., 0., 4., 4.], [2., 2., 6., 6.], [40., 0., 44., 4.]])
Mi, Mg = iou_matrix(A, B), giou_matrix(A, B)
assert Mi.shape == (3, 3) and Mg.shape == (3, 3)
# 与逐对的标量实现对拍
for i in range(3):
    for j in range(3):
        assert abs(Mi[i, j] - iou(tuple(A[i]), tuple(B[j]))) < 1e-9, (i, j)
        assert abs(Mg[i, j] - giou(tuple(A[i]), tuple(B[j]))) < 1e-9, (i, j)
assert abs(Mi[0, 0] - 1.0) < 1e-9 and Mi[0, 2] == 0.0
assert Mg[0, 2] < -0.8, '很远的不相交，GIoU 应接近 -1'
assert (Mg <= Mi + 1e-9).all(), 'GIoU <= IoU 处处成立'
# 大规模性能形状检查
big = iou_matrix(rng.uniform(0, 1, (100, 4)).cumsum(1),
                 rng.uniform(0, 1, (30, 4)).cumsum(1))
assert big.shape == (100, 30)
print('IoU 矩阵:\n', Mi)
print('GIoU 矩阵:\n', Mg)
print('✅ 练习 1 通过：DETR 的代价矩阵是 M x N 的，N 可以到 900 —— **必须向量化**')

## ✏️ 练习 2：完整的 DETR 匹配代价矩阵（含权重）

实现 `cost_matrix(prob, boxes, tgt_labels, tgt_boxes, w_cls, w_l1, w_giou)`：
- `prob` 是 `(N, K+1)` 的**概率**（已 softmax），`boxes` 是 `(N,4)` 的 cxcywh
- 分类项必须用 `-prob[:, c]`（**不是** `-log`）
- 返回 `(M, N)` 代价矩阵，且要向量化（至少 IoU 部分）

In [ ]:
def cost_matrix(prob, boxes, tgt_labels, tgt_boxes, w_cls=1.0, w_l1=5.0, w_giou=2.0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
probs = softmax(logits, -1)
Bp = np.asarray(pred_boxes); Bt = np.asarray(tgt_boxes)
Cm = cost_matrix(probs, Bp, tgt_labels, Bt)
Cref = detr_cost_matrix(logits, pred_boxes, tgt_labels, tgt_boxes)
assert Cm.shape == (3, N)
assert np.allclose(Cm, Cref, atol=1e-9), '与逐元素参考实现必须一致'
r_, c_ = hungarian(Cm)
assert dict(zip(r_.tolist(), c_.tolist())) == {0: 3, 1: 11, 2: 17}
print('默认权重 (1,5,2) 的匹配:', dict(zip(r_.tolist(), c_.tolist())))
print('代价矩阵三项的典型量级: cls in [%.3f,%.3f], L1 ~ %.3f, 1-GIoU ~ %.3f'
      % (-probs.max(), -probs.min(),
         float(np.abs(Bp[0] - Bt[0]).sum()),
         1 - giou(cxcywh_to_xyxy(Bp[0]), cxcywh_to_xyxy(Bt[0]))))

# —— 权重敏感性：构造一个「分类好但框差」vs「框好但分类差」的对峙 ——
Bt2 = np.array([[0.30, 0.30, 0.04, 0.04], [0.70, 0.30, 0.04, 0.04]])
Bp2 = np.array([[0.30, 0.30, 0.04, 0.04],    # q0: 框完美，分类平庸
                [0.34, 0.30, 0.04, 0.04]])   # q1: 框略差，分类极自信
pr2 = np.array([[0.34, 0.33, 0.33], [0.97, 0.02, 0.01]])   # (N=2, K+1=3)
lab2 = [0, 1]
for wc in [1.0, 20.0]:
    C2 = cost_matrix(pr2, Bp2, lab2, Bt2, w_cls=wc)
    _, cc = hungarian(C2)
    print('w_cls=%-5.1f -> GT0 认领 q%d, GT1 认领 q%d' % (wc, cc[0], cc[1]))
m_low = hungarian(cost_matrix(pr2, Bp2, lab2, Bt2, w_cls=1.0))[1]
m_high = hungarian(cost_matrix(pr2, Bp2, lab2, Bt2, w_cls=20.0))[1]
assert not np.array_equal(m_low, m_high), '权重改变了最优匹配'
print('⚠️  **同一组预测，只改分类权重，匹配就翻了** —— 匹配代价的权重不是无关紧要的超参。')
print('✅ 练习 2 通过：**匹配代价用概率、损失用 log** —— 这是最常被问的细节')

## ✏️ 练习 3：辅助损失与「层间匹配翻转率」

实现 `aux_total_loss(per_layer, tgt_labels, tgt_boxes, eos_coef)`：
- `per_layer` 是 `[(logits_l, boxes_l), ...]`，长度 = decoder 层数
- 每层**独立做一次匈牙利匹配**、算一份完整 Hungarian loss，等权相加
- 同时返回 `flip_rate`：相邻两层之间，**同一个 GT 被换了 query 认领**的比例
  （所有相邻层对上求平均）

In [ ]:
def aux_total_loss(per_layer, tgt_labels, tgt_boxes, eos_coef=0.1):
    # TODO: 返回 dict(total=..., per_layer=[...], flip_rate=...)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 造 6 层输出：模拟「逐层精修」—— 越靠后的层框越准、分类越自信
per_layer = []
for L in range(6):
    s = 0.05 * (6 - L) / 6
    bl = [jitter(b, s, s * 0.2) for b in pred_boxes]
    ll = logits.copy() + rng.normal(0, 0.3, logits.shape)
    for k, (gi, qi) in enumerate([(0, 3), (1, 11), (2, 17)]):
        bl[qi] = jitter(tgt_boxes[gi], 0.02 * (6 - L) / 6, 0.004 * (6 - L) / 6)
        ll[qi, tgt_labels[gi]] += 1.0 + 0.6 * L
    per_layer.append((ll, bl))

r = aux_total_loss(per_layer, tgt_labels, tgt_boxes)
assert set(r) >= {'total', 'per_layer', 'flip_rate'}
assert len(r['per_layer']) == 6
assert abs(r['total'] - sum(r['per_layer'])) < 1e-9, '等权相加'
assert 0.0 <= r['flip_rate'] <= 1.0
print('逐层损失:', ['%.3f' % v for v in r['per_layer']])
print('总损失 %.4f   层间匹配翻转率 %.3f' % (r['total'], r['flip_rate']))
assert r['per_layer'][-1] < r['per_layer'][0], '越靠后的层损失应更低（逐层精修）'
print('✅ 练习 3 通过：**翻转率是免费的诊断信号** —— 高翻转率意味着各层「认领」的目标不一致，')
print('   优化目标在抖。这正是 DN-DETR 用去噪 query 绕过匹配的动机（模块 04）。')

## ✏️ 练习 4：把梯度换到 `(cx, cy, w, h)` 参数化

DETR 的框头输出的是 `(cx, cy, w, h)`，而我们的 GIoU 梯度是对 `(x1,y1,x2,y2)` 的。
实现 `grad_giou_cxcywh(a_cxcywh, b_xyxy)`：用链式法则转换，并保证数值梯度校验通过。

> 提示：`x1 = cx - w/2, y1 = cy - h/2, x2 = cx + w/2, y2 = cy + h/2`，
> 所以雅可比是一个 4x4 常数矩阵。**面试里主动提到这一步是加分项。**

In [ ]:
def grad_giou_cxcywh(a_cxcywh, b_xyxy):
    # TODO: 先转 xyxy 求 grad_giou，再乘 d(xyxy)/d(cxcywh) 的雅可比
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
def giou_cxcywh(a_cxcywh, b_xyxy):
    return giou(cxcywh_to_xyxy(a_cxcywh), b_xyxy)

bad = 0
for _ in range(200):
    a = (float(rng.uniform(-1, 1)), float(rng.uniform(-1, 1)),
         float(rng.uniform(.3, 2)), float(rng.uniform(.3, 2)))
    bx = float(rng.uniform(-1, 1)); by = float(rng.uniform(-1, 1))
    b = (bx, by, bx + float(rng.uniform(.3, 2)), by + float(rng.uniform(.3, 2)))
    ga = grad_giou_cxcywh(a, b)
    gn = num_grad(giou_cxcywh, a, b)
    if not np.allclose(ga, gn, atol=2e-6):
        bad += 1
assert bad == 0, 'cxcywh 参数化的梯度校验失败 %d 例' % bad

a = (0.0, 0.0, 2.0, 1.0); b = (-0.5, -0.6, 1.5, 0.9)
print('d GIoU / d(cx, cy, w, h) 解析 =', grad_giou_cxcywh(a, b))
print('d GIoU / d(cx, cy, w, h) 数值 =', num_grad(giou_cxcywh, a, b))
print('✅ 练习 4 通过：真实训练里梯度还要再过一层 sigmoid（因为 cxcywh 被约束在 [0,1]），')
print('   所以完整链条是: GIoU -> xyxy -> cxcywh -> sigmoid -> 网络输出。')
print('   **白板题主动说出这一层，比只写 IoU 梯度高一个档次。**')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def iou_matrix(A, B):
    A = np.asarray(A, float); B = np.asarray(B, float)
    lt = np.maximum(A[:, None, :2], B[None, :, :2])
    rb = np.minimum(A[:, None, 2:], B[None, :, 2:])
    wh = np.clip(rb - lt, 0.0, None)                 # ← clamp
    I = wh[..., 0] * wh[..., 1]
    aA = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1])
    aB = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    U = aA[:, None] + aB[None, :] - I
    return I / (U + 1e-12)

def giou_matrix(A, B):
    A = np.asarray(A, float); B = np.asarray(B, float)
    lt = np.maximum(A[:, None, :2], B[None, :, :2])
    rb = np.minimum(A[:, None, 2:], B[None, :, 2:])
    wh = np.clip(rb - lt, 0.0, None)
    I = wh[..., 0] * wh[..., 1]
    aA = (A[:, 2] - A[:, 0]) * (A[:, 3] - A[:, 1])
    aB = (B[:, 2] - B[:, 0]) * (B[:, 3] - B[:, 1])
    U = aA[:, None] + aB[None, :] - I
    clt = np.minimum(A[:, None, :2], B[None, :, :2])
    crb = np.maximum(A[:, None, 2:], B[None, :, 2:])
    cwh = np.clip(crb - clt, 0.0, None)
    Ac = cwh[..., 0] * cwh[..., 1]
    return I / (U + 1e-12) - (Ac - U) / (Ac + 1e-12)

In [ ]:
# 练习 2 参考答案
def cost_matrix(prob, boxes, tgt_labels, tgt_boxes, w_cls=1.0, w_l1=5.0, w_giou=2.0):
    prob = np.asarray(prob, float)
    Bp = np.asarray(boxes, float); Bt = np.asarray(tgt_boxes, float)
    tgt_labels = np.asarray(tgt_labels, int)
    # 分类项：**用概率，不用 log** —— (M, N)
    c_cls = -prob[:, tgt_labels].T
    # L1 项：(M, N)
    c_l1 = np.abs(Bt[:, None, :] - Bp[None, :, :]).sum(-1)
    # GIoU 项：先转 xyxy 再用矩阵版
    def to_xyxy(B):
        return np.stack([B[:, 0] - B[:, 2] / 2, B[:, 1] - B[:, 3] / 2,
                         B[:, 0] + B[:, 2] / 2, B[:, 1] + B[:, 3] / 2], -1)
    c_gi = 1.0 - giou_matrix(to_xyxy(Bt), to_xyxy(Bp))
    return w_cls * c_cls + w_l1 * c_l1 + w_giou * c_gi

In [ ]:
# 练习 3 参考答案
def aux_total_loss(per_layer, tgt_labels, tgt_boxes, eos_coef=0.1):
    losses, matches = [], []
    for lg, bx in per_layer:
        r = hungarian_loss(lg, bx, tgt_labels, tgt_boxes, eos_coef)
        losses.append(r['total']); matches.append(r['match'])
    flips = 0; pairs = 0
    for k in range(len(matches) - 1):
        for gi in matches[k]:
            pairs += 1
            if matches[k][gi] != matches[k + 1].get(gi, -1):
                flips += 1
    return dict(total=float(sum(losses)), per_layer=losses,
                flip_rate=flips / max(pairs, 1))

In [ ]:
# 练习 4 参考答案
def grad_giou_cxcywh(a_cxcywh, b_xyxy):
    g_xyxy = grad_giou(cxcywh_to_xyxy(a_cxcywh), b_xyxy)
    # x1=cx-w/2, y1=cy-h/2, x2=cx+w/2, y2=cy+h/2
    # J[k, m] = d xyxy[k] / d cxcywh[m]
    J = np.array([[1., 0., -0.5, 0.],
                  [0., 1., 0., -0.5],
                  [1., 0., +0.5, 0.],
                  [0., 1., 0., +0.5]])
    return g_xyxy @ J

---
## 🧪 真实工程胶囊：可直接抄进项目的集合损失实现要点

In [ ]:
RECIPE = r'''
# ============ DETR 集合损失：工程 checklist（PyTorch 伪代码 + 真实参数） ============

# ---- 1. 匹配（no_grad！分类项用概率） ----
@torch.no_grad()
def matcher(outputs, targets, w_cls=1.0, w_l1=5.0, w_giou=2.0):
    bs, nq = outputs["pred_logits"].shape[:2]
    out_prob = outputs["pred_logits"].flatten(0, 1).softmax(-1)   # <- 概率，不是 log
    out_bbox = outputs["pred_boxes"].flatten(0, 1)                # cxcywh, 归一化
    tgt_ids  = torch.cat([t["labels"] for t in targets])
    tgt_bbox = torch.cat([t["boxes"]  for t in targets])
    cost_cls  = -out_prob[:, tgt_ids]                             # (bs*nq, sum_M)
    cost_l1   = torch.cdist(out_bbox, tgt_bbox, p=1)
    cost_giou = -generalized_box_iou(box_cxcywh_to_xyxy(out_bbox),
                                     box_cxcywh_to_xyxy(tgt_bbox))
    C = (w_l1*cost_l1 + w_cls*cost_cls + w_giou*cost_giou).view(bs, nq, -1).cpu()
    sizes = [len(t["boxes"]) for t in targets]
    return [linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))]
    # ^ 官方用 scipy.optimize.linear_sum_assignment；**匹配在 CPU 上做，是训练吞吐瓶颈之一**

# ---- 2. 损失（可导！分类项用 log；只对匹配上的算框损失） ----
empty_weight = torch.ones(num_classes + 1); empty_weight[-1] = 0.1   # eos_coef
loss_ce = F.cross_entropy(src_logits.transpose(1, 2), target_classes, empty_weight)
loss_bbox = F.l1_loss(src_boxes, target_boxes, reduction="sum") / num_boxes
loss_giou = (1 - torch.diag(generalized_box_iou(
        box_cxcywh_to_xyxy(src_boxes), box_cxcywh_to_xyxy(target_boxes)))).sum() / num_boxes
# num_boxes = 全 GPU 上的 GT 总数（**必须 all_reduce，否则多卡不等价**）

# ---- 3. 权重（DETR 官方 detr/main.py 默认值） ----
WEIGHT_DICT = {"loss_ce": 1, "loss_bbox": 5, "loss_giou": 2}
EOS_COEF    = 0.1
AUX_LOSS    = True     # 6 层 decoder 各来一份，权重与最后一层相同
# 辅助损失展开: {"loss_ce_0": 1, "loss_bbox_0": 5, ...} 共 6 组

# ---- 4. 换成 focal 版（Deformable DETR / DINO / RT-DETR 的主流） ----
# · 输出层: K 个 sigmoid，**没有 no-object 类**
# · cls loss: sigmoid_focal_loss(alpha=0.25, gamma=2.0)，weight = 2.0
# · 匹配代价的分类项也换成 focal 形式:
#     neg = (1-a) * p**g * (-log(1-p));  pos = a * (1-p)**g * (-log p)
#     cost_cls = pos[:, tgt_ids] - neg[:, tgt_ids]
# · num_queries: 100 -> 300 (Deformable) / 900 (DINO)
# · 推理: 在 N*K 个 (query, class) 对里取 top-k，**同一 query 可出多类**

# ---- 5. 上线前必查的 5 个数值 bug ----
# [ ] U = Aa + Ab - I，不是 Aa + Ab
# [ ] 交集 wh 必须 clamp(min=0)（不 clamp -> 不相交时出现「假交集」）
# [ ] 框在算 GIoU 前必须满足 x2>=x1, y2>=y1（否则 GIoU 无定义，官方代码里有 assert）
# [ ] num_boxes 要跨卡 all_reduce 并 clamp(min=1)
# [ ] 归一化坐标 vs 像素坐标：换表示必须同步改 lambda_L1（差 640 倍！）
'''
print(RECIPE)
for key in ['no_grad', 'softmax(-1)', 'empty_weight', 'eos_coef', 'all_reduce',
            'clamp(min=0)', 'sigmoid_focal_loss', 'loss_giou']:
    assert key in RECIPE, key
print('✅ 配方覆盖：匹配(no_grad/概率) / 损失(log/eos_coef) / 权重 / focal 版 / 5 个数值 bug')

### 小结

- **匹配代价 ≠ 训练损失**。匹配只需排序 → 用有界的 `-p` 让三项量纲可比；
  损失需要梯度 → 用 `-log p`。**这是 DETR 损失最常被追问的细节。**
- **背景 query 占 97%**（TSR 场景更极端）。不给 no-object 降权，
  模型会立刻收敛到「全说背景」。`eos_coef=0.1` 是刻意保留背景优势的工作点，
  **不是配平**。而 focal loss 版把这个超参彻底消掉了。
- **L1 与 GIoU 各补一个洞**：L1 对尺度不公平（8×8 和 256×256 的框，2px 偏移的 L1 完全相同，
  1−IoU 却差 20 倍），IoU 在不相交时梯度**精确为 0**。
  8–32 px 的交通标志正好落在「L1 最不公平、GIoU 最关键」的区间。
- **IoU 解析梯度的三个得分点**：`U = Aa+Ab−I`、clamp 产生的截断乘子 `1[iw>0]1[ih>0]`、
  `max/min` 产生的指示函数（外接框方向相反）。加分项：再链式到 `(cx,cy,w,h)` 和 sigmoid。
- **GIoU→DIoU→CIoU 是一条补洞链**：不相交无梯度 → 包含关系退化 → 宽高比无约束。
  DETR 用 GIoU 而非 CIoU，是因为它已经有 L1 项覆盖了中心与宽高。
- **辅助损失**（每层 decoder 各算一份）值约 +2 AP，还顺带给出两个红利：
  层间匹配翻转率这个免费诊断信号，以及 RT-DETR「同一份权重多档速度」的部署能力。

下一站：**模块 03 · Object query 与交叉注意力** —— 谁在认领目标，以及它们怎么互相协商去重。